In [1]:
# ==============================================================================
# ONCOPREDICT - MODEL TRAINING & COMPRESSED EXPORT SCRIPT (SCIKIT-LEARN)
# ==============================================================================

import os
import glob
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score
import warnings 
warnings.filterwarnings('ignore')

# ------------------------------------------------------------------------------
# STEP 1: DIRECT DATASET LOADING
# ------------------------------------------------------------------------------
print(" Loading breast cancer prediction dataset...")

csv_path = '/kaggle/input/datasets/mobeenfatimah/breast-cancer-prediction-dataset-10000-patient/breast_cancer_prediction.csv'

if not os.path.exists(csv_path):
    kaggle_input = '/kaggle/input'
    all_csvs = glob.glob(os.path.join(kaggle_input, '**/*.csv'), recursive=True)
    valid_csvs = [f for f in all_csvs if 'data_dictionary.csv' not in f]
    csv_path = max(valid_csvs, key=os.path.getsize)

print(f" Loaded dataset from: {csv_path}")

df = pd.read_csv(csv_path)

# Clean column headers
df.columns = df.columns.str.strip()
print(f" Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")

# ------------------------------------------------------------------------------
# STEP 2: TARGET & FEATURE SEPARATION
# ------------------------------------------------------------------------------
target_col = 'Cancer'

if target_col not in df.columns:
    raise ValueError(f"Target column '{target_col}' not found. Columns: {list(df.columns)}")

print(f" Target column set to: '{target_col}'")

# Drop Patient_ID and data leak column (Cancer_Stage is determined post-diagnosis)
drop_cols = ['Patient_ID', 'Cancer_Stage']
cols_to_drop = [c for c in drop_cols if c in df.columns]

if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"🗑️ Dropped metadata & target-leak columns: {cols_to_drop}")

# Separate Features (X) and Target (y)
X = df.drop(columns=[target_col])
y = df[target_col]

# Handle Categorical Target Mapping if necessary
if y.dtype == 'object' or isinstance(y.iloc[0], str):
    mapping = {'Malignant': 1, 'Benign': 0, 'M': 1, 'B': 0, 'Yes': 1, 'No': 0}
    y = y.map(mapping).fillna(y)

y = y.astype(int)

# One-Hot Encode categorical features (e.g. Gender, Smoking, Biopsy_Result, etc.)
object_cols = X.select_dtypes(include=['object']).columns
if len(object_cols) > 0:
    print(f"⚙️ One-Hot Encoding categorical features: {list(object_cols)}")
    X = pd.get_dummies(X, columns=object_cols, drop_first=True)

feature_list = list(X.columns)
print(f" Final Model Features ({len(feature_list)} total): {feature_list}")

from sklearn.impute import SimpleImputer

# ------------------------------------------------------------------------------
# STEP 3: TRAIN / TEST SPLIT & PREPROCESSING (IMPUTATION + SCALING)
# ------------------------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Impute NaNs using median strategy (works for numerical & 0/1 dummies)
imputer = SimpleImputer(strategy='median')
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

# Scale features after imputation
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

# ------------------------------------------------------------------------------
# STEP 4: MODEL TRAINING (SCIKIT-LEARN LOGISTIC REGRESSION)
# ------------------------------------------------------------------------------
print("\n⚡ Training Scikit-Learn Logistic Regression Classifier...")
model = LogisticRegression(
    max_iter=1000,
    C=1.0,
    random_state=42,
    solver='lbfgs'
)
model.fit(X_train_scaled, y_train)

# ------------------------------------------------------------------------------
# STEP 5: EVALUATION METRICS
# ------------------------------------------------------------------------------
preds = model.predict(X_test_scaled)
probs = model.predict_proba(X_test_scaled)[:, 1]

print("\n" + "="*50)
print("             MODEL PERFORMANCE REPORT             ")
print("="*50)
print(f"Accuracy Score: {accuracy_score(y_test, preds) * 100:.2f}%")
print(f"ROC-AUC Score : {roc_auc_score(y_test, probs):.4f}")
print("\nClassification Report:\n", classification_report(y_test, preds))
print("="*50)

# ------------------------------------------------------------------------------
# STEP 6: COMPRESSED EXPORT FOR VERCEL DEPLOYMENT
# ------------------------------------------------------------------------------
print("\n Exporting compressed model artifacts...")

# Save model, scaler, and exact column order
joblib.dump(model, 'model.joblib', compress=9)
joblib.dump(scaler, 'scaler.joblib', compress=9)
joblib.dump(feature_list, 'feature_names.joblib')

for file in ['model.joblib', 'scaler.joblib', 'feature_names.joblib']:
    size_kb = os.path.getsize(file) / 1024
    print(f" - {file}: {size_kb:.2f} KB")

print("\n Export Complete! Download `model.joblib`, `scaler.joblib`, and `feature_names.joblib` into your Flask project's model directory.")

 Loading breast cancer prediction dataset...
 Loaded dataset from: /kaggle/input/datasets/mobeenfatimah/breast-cancer-prediction-dataset-10000-patient/breast_cancer_prediction.csv
 Dataset Shape: 10000 rows, 23 columns
 Target column set to: 'Cancer'
🗑️ Dropped metadata & target-leak columns: ['Patient_ID', 'Cancer_Stage']
⚙️ One-Hot Encoding categorical features: ['Gender', 'Family_History', 'Smoking', 'Alcohol_Consumption', 'Physical_Activity', 'Hormone_Therapy', 'Menopause_Status', 'Genetic_Mutation', 'Lymph_Node_Involvement', 'Mammogram_Result', 'Biopsy_Result', 'Diabetes', 'Breastfeeding_History']
 Final Model Features (23 total): ['Age', 'BMI', 'Tumor_Size_cm', 'Blood_Pressure', 'Cholesterol', 'Exercise_Days_Per_Week', 'Annual_Income_USD', 'Gender_Male', 'Family_History_Yes', 'Smoking_Yes', 'Alcohol_Consumption_Yes', 'Physical_Activity_Low', 'Physical_Activity_Moderate', 'Hormone_Therapy_Yes', 'Menopause_Status_Pre', 'Genetic_Mutation_Positive', 'Lymph_Node_Involvement_Yes', 'Mam